# Facial Emotion Recognition Ã¢â‚¬â€ Research Notebook

**Research notebook** driven by project config.yaml for dataset paths, model selection, preprocessing, and training settings.

**Datasets supported:**
- `AffectNet+` Ã¢â‚¬â€ JSON annotation format (`human_annotated/{train_set,validation_set}/{images,annotations}`)
- `FER2013` Ã¢â‚¬â€ Folder-based (`train/<class>/`, `test/<class>/`)
- `FER2013 CSV` Ã¢â‚¬â€ CSV format (`fer2013.csv`)
- `CK+` Ã¢â‚¬â€ CSV format (`ckextended.csv`)

**Pipeline:** Config Ã¢â€ â€™ Dataset Paths Ã¢â€ â€™ Data Loading Ã¢â€ â€™ Preprocessing Ã¢â€ â€™ Model Training Ã¢â€ â€™ Evaluation Ã¢â€ â€™ Results

> Edit config.yaml to change shared settings. The notebook currently loads the enabled datasets and runs the selected baseline model.

## 0. Configuration Ã¢â‚¬â€ Edit This Section

In [ ]:
import os, sys, json, warnings
from pathlib import Path
import yaml

_here = Path.cwd().resolve()
PROJECT_ROOT = _here.parent if _here.name == "notebooks" else _here
CONFIG_PATH = PROJECT_ROOT / "config.yaml"
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Could not find project config: {CONFIG_PATH}")
with CONFIG_PATH.open(encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

def _project_path(value):
    p = Path(value)
    return p if p.is_absolute() else (PROJECT_ROOT / p).resolve()

ds_cfg = CONFIG["datasets"]
PATHS = {
    "affectnet": _project_path(ds_cfg["affectnet"]["root"]),
    "fer2013_folders": _project_path(ds_cfg["fer2013"]["root"]),
    "fer2013_csv": _project_path(ds_cfg["fer2013"]["csv"]["file"]),
    "ckplus": _project_path(ds_cfg["ckplus"]["root"]) / ds_cfg["ckplus"]["csv_extended"]["file"],
    "ravdess": _project_path(ds_cfg["ravdess"]["root"]),
}

ENABLED_DATASETS = []
if ds_cfg["affectnet"].get("enabled", True):
    ENABLED_DATASETS.append("affectnet")
if ds_cfg["fer2013"].get("enabled", True):
    fmt = ds_cfg["fer2013"].get("format", "folders")
    if fmt == "folders":
        ENABLED_DATASETS.append("fer2013_folders")
    elif fmt == "csv":
        ENABLED_DATASETS.append("fer2013_csv")
    else:
        ENABLED_DATASETS.extend(["fer2013_folders", "fer2013_csv"])
if ds_cfg["ckplus"].get("enabled", True):
    ENABLED_DATASETS.append("ckplus")
if ds_cfg["ravdess"].get("enabled", True):
    ENABLED_DATASETS.append("ravdess")

CANONICAL_CLASSES = list(CONFIG["label_mapping"]["canonical_classes"])
ENABLED_MODELS = {
    name: dict(spec)
    for name, spec in CONFIG["models"]["architectures"].items()
    if spec.get("enabled", True)
}

t = CONFIG["training"]
TRAINING = {
    "mode": t.get("mode", "head_then_full"),
    "epochs": t["epochs"],
    "head_epochs": t.get("head_epochs", 2),
    "batch_size": CONFIG["environment"]["batch_size"],
    "lr": t["optimizer"]["lr"],
    "backbone_lr": t["optimizer"].get("backbone_lr") or t["optimizer"]["lr"],
    "weight_decay": t["optimizer"]["weight_decay"],
    "optimizer": t["optimizer"]["name"],
    "momentum": t["optimizer"].get("momentum", 0.9),
    "nesterov": t["optimizer"].get("nesterov", True),
    "scheduler": t["scheduler"]["name"],
    "warmup_epochs": t["scheduler"].get("warmup_epochs", 1),
    "min_lr_ratio": t["scheduler"].get("min_lr_ratio", 0.01),
    "early_stopping_enabled": t.get("early_stopping", {}).get("enabled", True),
    "early_stopping_patience": t.get("early_stopping", {}).get("patience", 6),
    "early_stopping_min_delta": t.get("early_stopping", {}).get("min_delta", 0.001),
    "selection_metric": t.get("selection_metric", "val_macro_f1"),
    "label_smoothing": CONFIG["loss"].get("label_smoothing", 0.0),
    "loss": CONFIG["loss"]["name"],
    "focal_gamma": CONFIG["loss"].get("focal_gamma", 2.0),
    "focal_use_class_weights": CONFIG["loss"].get("focal_use_class_weights", False),
    "gradient_clip": CONFIG["environment"]["gradient_clipping"]["max_norm"] if CONFIG["environment"]["gradient_clipping"]["enabled"] else 0.0,
    "seed": CONFIG["environment"]["seed"],
    "max_samples_per_dataset": t.get("max_samples_per_dataset", None),
    "num_workers": 0,  # 0 is recommended on Windows to avoid DataLoader spawn issues
}

PREPROCESSING = {
    "color_mode": CONFIG["preprocessing"]["color_mode"],
    "normalization": CONFIG["preprocessing"]["normalization"],
    "augmentation": CONFIG["augmentation"]["preset"],
}

def _project_path_out(value):
    p = Path(value)
    return p if p.is_absolute() else (PROJECT_ROOT / p).resolve()

CACHE_DIR = _project_path_out(CONFIG["paths"]["cache_dir"])
OUTPUT_DIR = _project_path_out(CONFIG["paths"]["output_dir"])
CHECKPOINT_DIR = _project_path_out(CONFIG["paths"]["checkpoint_dir"])
RESULTS_DIR = _project_path_out(CONFIG["paths"]["results_dir"])
LOG_DIR = _project_path_out(CONFIG["paths"]["log_dir"])
for d in [CACHE_DIR, OUTPUT_DIR, CHECKPOINT_DIR, RESULTS_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

_baselines = [x for x in CONFIG.get("experiments", {}).get("definitions", []) if x.get("group") == "A"]
QUICK_MODEL = next((x["model"] for x in _baselines if x.get("model") in ENABLED_MODELS), next(iter(ENABLED_MODELS), None))
DEVICE = CONFIG["environment"]["device"]

print("Configuration loaded from:", CONFIG_PATH)
print("Project root:", PROJECT_ROOT)
print("Enabled datasets:", ENABLED_DATASETS)
print("Default model:", QUICK_MODEL)
print("Training mode:", TRAINING["mode"])


## 1. Environment & Dependencies Check

In [ ]:
import subprocess
import importlib.util

REQUIRED = ["torch", "torchvision", "timm", "numpy", "pandas", "PIL", "sklearn", "matplotlib", "cv2"]
missing = [pkg for pkg in REQUIRED if not importlib.util.find_spec(pkg)]
if missing:
    print(f"Missing packages: {missing}")
    print("Install: pip install torch torchvision timm numpy pandas pillow scikit-learn matplotlib opencv-python")
else:
    print("All dependencies found.")

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

if DEVICE == "auto":
    if torch.cuda.is_available():
        try:
            torch.zeros(1, device="cuda")
            device = torch.device("cuda")
        except Exception as e:
            print(f"Notice: GPU detected ({torch.cuda.get_device_name(0)}) but current PyTorch build lacks sm_120 kernels. Falling back to CPU.")
            device = torch.device("cpu")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device(DEVICE)

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")


## 2. Validate Dataset Paths

In [ ]:
def validate_paths():
    checks = {
        "affectnet": {
            "required": [
                PATHS["affectnet"] / "train_set" / "images",
                PATHS["affectnet"] / "train_set" / "annotations",
                PATHS["affectnet"] / ds_cfg["affectnet"]["json_per_image"]["val_dir"] / ds_cfg["affectnet"]["json_per_image"]["images_subdir"],
                PATHS["affectnet"] / ds_cfg["affectnet"]["json_per_image"]["val_dir"] / ds_cfg["affectnet"]["json_per_image"]["annotations_subdir"],
            ],
            "format": "AffectNet+ JSON: human_annotated/{train_set,validation_set}/{images,annotations}"
        },
        "fer2013_folders": {
            "required": [PATHS["fer2013_folders"] / "train", PATHS["fer2013_folders"] / "test"],
            "format": "FER2013 folders: train/<class>/*.jpg, test/<class>/*.jpg"
        },
        "fer2013_csv": {
            "required": [PATHS["fer2013_csv"]],
            "format": "FER2013 CSV: fer2013.csv (emotion, pixels, Usage)"
        },
        "ravdess": {
            "required": [PATHS["ravdess"]],
            "format": "RAVDESS video: actor folders containing labelled .mp4 clips"
        },
        "ckplus": {            "required": [PATHS["ckplus"]],
            "format": "CK+ CSV: ckextended.csv (emotion, pixels, Usage)"
        },
    }
    results, available = [], []
    for name, info in checks.items():
        missing = [str(p) for p in info["required"] if not Path(p).exists()]
        status = "OK" if not missing else "MISSING"
        if not missing and name in ENABLED_DATASETS:
            available.append(name)
        results.append({"dataset": name, "enabled": name in ENABLED_DATASETS,
                        "status": status, "format": info["format"],
                        "missing": "; ".join(missing) if missing else ""})
    return pd.DataFrame(results), available

path_report, AVAILABLE = validate_paths()
display(path_report)
print(f"\nAvailable & enabled: {AVAILABLE}")

## 3. Dataset Loaders

In [ ]:
import json as _json
import random

CLASS2IDX = {c: i for i, c in enumerate(CANONICAL_CLASSES)}
IDX2CLASS  = {i: c for c, i in CLASS2IDX.items()}

# AffectNet+ human-label codes (0-7)
AFFECTNET_LABEL_MAP = {0: "neutral", 1: "happy", 2: "sad", 3: "surprise",
                       4: "fear", 5: "disgust", 6: "angry", 7: None}

# FER2013 CSV emotion codes
FER2013_CSV_MAP = {0: "angry", 1: "disgust", 2: "fear", 3: "happy",
                   4: "sad", 5: "surprise", 6: "neutral"}

# CK+ CSV emotion codes: note code 1 is 'angry' (canonical taxonomy)
CKPLUS_CSV_MAP = {0: "neutral", 1: "angry", 2: None, 3: "disgust",
                  4: "fear", 5: "happy", 6: "sad", 7: "surprise"}


def load_affectnet(root: Path, split: str = "train", max_samples=None):
    split_name = (ds_cfg["affectnet"]["json_per_image"]["train_dir"] if split == "train" else ds_cfg["affectnet"]["json_per_image"]["val_dir"])
    split_dir  = root / split_name
    images_dir = split_dir / ds_cfg["affectnet"]["json_per_image"]["images_subdir"]
    annot_dir  = split_dir / ds_cfg["affectnet"]["json_per_image"]["annotations_subdir"]
    if not annot_dir.exists():
        return []
    samples, skipped = [], 0
    ann_files = sorted(annot_dir.glob("*.json"))
    if max_samples:
        ann_files = ann_files[:max_samples * 2]
    for ann_file in ann_files:
        img_path = images_dir / (ann_file.stem + ".jpg")
        if not img_path.exists():
            skipped += 1; continue
        try:
            ann = _json.loads(ann_file.read_text())
            lc  = ann.get(ds_cfg["affectnet"]["json_per_image"]["label_field"], -1)
            label_code = int(lc[0]) if isinstance(lc, (list, tuple)) else int(lc)
        except Exception:
            skipped += 1; continue
        canonical = AFFECTNET_LABEL_MAP.get(label_code)
        if canonical is None or canonical not in CLASS2IDX:
            skipped += 1; continue
        samples.append((str(img_path), CLASS2IDX[canonical]))
        if max_samples and len(samples) >= max_samples:
            break
    print(f"  [AffectNet {split}] {len(samples)} loaded, {skipped} skipped")
    return samples


def load_fer2013_folders(root: Path, split: str = "train", max_samples=None):
    FOLDER_MAP = {"angry": "angry", "disgust": "disgust", "fear": "fear",
                  "happy": "happy", "sad": "sad", "surprise": "surprise", "neutral": "neutral"}
    split_dir = root / split
    if not split_dir.exists():
        return []
    samples = []
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        canonical = FOLDER_MAP.get(class_dir.name.lower())
        if canonical is None:
            continue
        for img_file in sorted(class_dir.glob("*")):
            if img_file.suffix.lower() in (".jpg", ".jpeg", ".png"):
                samples.append((str(img_file), CLASS2IDX[canonical]))
    if max_samples and len(samples) > max_samples:
        random.seed(TRAINING["seed"])
        samples = random.sample(samples, max_samples)
    print(f"  [FER2013-Folders {split}] {len(samples)} loaded")
    return samples


def _load_pixel_csv(csv_path: Path, label_map: dict, split: str = "train",
                    train_usage="Training", test_usage="PrivateTest",
                    max_samples=None, name="CSV"):
    df = pd.read_csv(csv_path)
    if "Usage" in df.columns:
        usage = train_usage if split == "train" else test_usage
        df = df[df["Usage"] == usage]
    elif split == "test":
        split_idx = int(len(df) * 0.8)
        df = df.iloc[split_idx:]
    else:
        split_idx = int(len(df) * 0.8)
        df = df.iloc[:split_idx]
    samples, skipped = [], 0
    for _, row in df.iterrows():
        canonical = label_map.get(int(row["emotion"]))
        if canonical is None or canonical not in CLASS2IDX:
            skipped += 1; continue
        try:
            pxs = list(map(int, str(row["pixels"]).split()))
            sz  = int(np.sqrt(len(pxs)))
            if sz * sz != len(pxs):
                skipped += 1; continue
            img = np.array(pxs, dtype=np.uint8).reshape(sz, sz)
        except Exception:
            skipped += 1; continue
        samples.append((img, CLASS2IDX[canonical]))
        if max_samples and len(samples) >= max_samples:
            break
    print(f"  [{name} {split}] {len(samples)} loaded, {skipped} skipped")
    return samples


def load_fer2013_csv(csv_path, split="train", max_samples=None):
    usage_vals = ds_cfg["fer2013"]["csv"].get("usage_values", {})
    train_u = usage_vals.get("train", "Training")
    test_u  = usage_vals.get("test", "PrivateTest")
    return _load_pixel_csv(csv_path, FER2013_CSV_MAP, split,
                           train_u, test_u, max_samples, "FER2013-CSV")


def load_ckplus_csv(csv_path, split="train", max_samples=None):
    usage_vals = ds_cfg["ckplus"]["csv_extended"].get("usage_values", {})
    train_u = usage_vals.get("train", ds_cfg["ckplus"]["csv_extended"].get("train_usage", "Training"))
    test_u  = usage_vals.get("test", ds_cfg["ckplus"]["csv_extended"].get("test_usage", "PrivateTest"))
    return _load_pixel_csv(csv_path, CKPLUS_CSV_MAP, split,
                           train_u, test_u, max_samples, "CK+")


def load_ravdess(root: Path, split="train", max_samples=None):
    """Extract cached facial frames from RAVDESS video-only clips with actor-held-out splits."""
    import cv2
    cfg = ds_cfg["ravdess"]
    split_cfg = CONFIG["splits"]["ravdess"]
    fractions = split_cfg["holdout_fractions"]
    actors = sorted({
        int(p.parts[-2].replace("Actor_", ""))
        for p in root.glob(cfg["video_glob"])
        if p.is_file() and p.suffix.lower() == ".mp4" and "Actor_" in p.parts[-2]
    })
    if not actors:
        return []
    rng = random.Random(CONFIG["splits"]["seed"])
    shuffled = actors[:]
    rng.shuffle(shuffled)
    n_train = max(1, round(len(shuffled) * fractions["train"]))
    n_val = round(len(shuffled) * fractions["val"])
    actor_sets = {
        "train": set(shuffled[:n_train]),
        "val": set(shuffled[n_train:n_train+n_val]),
        "test": set(shuffled[n_train+n_val:]),
    }
    heldout = actor_sets["val"] | actor_sets["test"]
    allowed_actors = actor_sets["train"] if split == "train" else heldout
    video_files = sorted(root.glob(cfg["video_glob"]))
    frame_cfg = cfg["frame_extraction"]
    frame_root = CACHE_DIR / "ravdess_frames"
    frame_root.mkdir(parents=True, exist_ok=True)
    samples, seen_clips = [], 0
    for video in video_files:
        if not video.is_file() or video.suffix.lower() != ".mp4":
            continue
        parts = video.stem.split("-")
        if len(parts) != 7:
            continue
        modality, channel, emotion, intensity, statement, repetition, actor_code = parts
        actor = int(actor_code)
        if actor not in allowed_actors:
            continue
        if modality not in cfg["modalities"] or channel not in cfg["vocal_channels"] or intensity not in cfg["intensities"]:
            continue
        emotion_name = cfg["label_names"].get(int(emotion))
        canonical = CONFIG["label_mapping"]["per_dataset"]["ravdess"].get(emotion_name)
        if canonical is None or canonical not in CLASS2IDX:
            continue
        clip_key = video.stem
        clip_dir = frame_root / clip_key
        existing = sorted(clip_dir.glob("frame_*.jpg")) if clip_dir.exists() else []
        if not existing:
            cap = cv2.VideoCapture(str(video))
            if not cap.isOpened():
                cap.release()
                continue
            fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
            n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
            duration = n_frames / fps if n_frames else 0
            start = frame_cfg["skip_start_seconds"]
            end = max(start, duration - frame_cfg["skip_end_seconds"])
            times = []
            t = start
            while t <= end and len(times) < frame_cfg["max_frames_per_clip"]:
                times.append(t)
                t += frame_cfg["sample_every_seconds"]
            clip_dir.mkdir(parents=True, exist_ok=True)
            for frame_idx, seconds in enumerate(times):
                cap.set(cv2.CAP_PROP_POS_MSEC, seconds * 1000)
                ok, frame = cap.read()
                if not ok:
                    continue
                h, w = frame.shape[:2]
                side = min(h, w)
                x0, y0 = (w-side)//2, (h-side)//2
                frame = frame[y0:y0+side, x0:x0+side]
                frame = cv2.resize(frame, (CONFIG["preprocessing"]["cache_resolution"], CONFIG["preprocessing"]["cache_resolution"]))
                out = clip_dir / f"frame_{frame_idx:02d}.jpg"
                cv2.imwrite(str(out), frame)
            cap.release()
            existing = sorted(clip_dir.glob("frame_*.jpg"))
        samples.extend((str(frame), CLASS2IDX[canonical]) for frame in existing)
        seen_clips += 1
        if max_samples and seen_clips >= max_samples:
            break
    print(f"  [RAVDESS {split}] {len(samples)} frames from {seen_clips} clips; actor holdout applied")
    return samples

print("Dataset loaders defined.")


## 4. Load All Datasets

In [ ]:
# EfficientNet-B0 base-dataset run: AffectNet+ only.
# AffectNet+ train_set trains the model; validation_set is used for per-epoch
# model selection and the reported held-out score. Other datasets are excluded.
max_s = TRAINING["max_samples_per_dataset"]
if "affectnet" not in AVAILABLE:
    raise RuntimeError("AffectNet+ is not available. Check its configured dataset path.")

ALL_TRAIN = []
ALL_TEST = []  # AffectNet+ validation split (used as validation during training)
DATASET_STATS = []

aff_train = load_affectnet(PATHS["affectnet"], "train", max_s)
aff_val = load_affectnet(PATHS["affectnet"], "val", max_s // 5 if max_s else None)
ALL_TRAIN = [(s, l, "affectnet") for s, l in aff_train]
ALL_TEST = [(s, l, "affectnet") for s, l in aff_val]
DATASET_STATS.append({"dataset": "affectnet", "train": len(aff_train), "validation": len(aff_val)})

if not ALL_TRAIN or not ALL_TEST:
    raise RuntimeError("AffectNet+ train or validation split loaded no usable samples.")

print(f"AffectNet+-only run | train: {len(ALL_TRAIN)} | validation: {len(ALL_TEST)}")
display(pd.DataFrame(DATASET_STATS))


## 5. Class Distribution

In [ ]:
if ALL_TRAIN:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, data, title in [
        (axes[0], ALL_TRAIN, "Train"),
        (axes[1], ALL_TEST,  "Test")
    ]:
        labels = [l for _, l, _ in data]
        counts = [labels.count(i) for i in range(len(CANONICAL_CLASSES))]
        bars = ax.bar(CANONICAL_CLASSES, counts, color=plt.cm.Set2.colors[:7])
        ax.set_title(f"{title} Class Distribution (N={sum(counts)})")
        ax.set_xlabel("Emotion"); ax.set_ylabel("Samples")
        ax.tick_params(axis='x', rotation=30)
        for bar, count in zip(bars, counts):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                    str(count), ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "class_distribution.png", dpi=100, bbox_inches='tight')
    plt.show()

    records = [{"dataset": ds, "emotion": CANONICAL_CLASSES[l]} for _, l, ds in ALL_TRAIN]
    pivot = pd.DataFrame(records).pivot_table(
        index="dataset", columns="emotion", aggfunc="size", fill_value=0)
    display(pivot)

## 6. PyTorch Dataset and DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


class SafeRandomErasing:
    """Wrapper around transforms.RandomErasing to ensure tensor input."""
    def __init__(self, p=0.25, scale=(0.02, 0.1)):
        self.eraser = transforms.RandomErasing(p=p, scale=scale)

    def __call__(self, img):
        if not isinstance(img, torch.Tensor):
            img = transforms.functional.to_tensor(img)
        return self.eraser(img)


def get_transforms(input_size, split, augmentation="standard", normalization="imagenet"):
    norm = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD) if normalization == "imagenet" \
           else transforms.Lambda(lambda x: x)
    aug = {
        "none": [],
        "light": [
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomRotation(8),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
        ],
        "standard": [
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomResizedCrop(input_size, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
            transforms.RandomRotation(10),
            transforms.RandomAffine(0, translate=(0.05, 0.05), scale=(0.95, 1.05), shear=3),
            transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.1),
        ],
        "mix": [
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomResizedCrop(input_size, scale=(0.85, 1.0)),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
        ],
    }.get(augmentation, [])

    if split == "train":
        # 1. PIL transforms first (resize and augmentations)
        tf_list = [transforms.Resize((input_size, input_size))] + aug
        # 2. Convert to Tensor
        tf_list.append(transforms.ToTensor())
        # 3. Tensor-only transforms (RandomErasing, Normalize)
        if augmentation == "standard":
            tf_list.append(SafeRandomErasing(p=0.25, scale=(0.02, 0.1)))
        tf_list.append(norm)
    else:
        tf_list = [transforms.Resize((input_size, input_size)), transforms.ToTensor(), norm]

    return transforms.Compose(tf_list)


class FERDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample, label = self.samples[idx]
        if isinstance(sample, np.ndarray):
            img = Image.fromarray(sample, mode='L').convert('RGB')
        elif isinstance(sample, str):
            try:
                img = Image.open(sample).convert('RGB')
            except Exception:
                img = Image.fromarray(np.zeros((48, 48, 3), dtype=np.uint8))
        else:
            img = Image.fromarray(np.zeros((48, 48, 3), dtype=np.uint8))
        if self.transform:
            img = self.transform(img)
        if not isinstance(img, torch.Tensor):
            img = transforms.functional.to_tensor(img)
        return img, label


def make_loaders(train_data, test_data, input_size, batch_size,
                 augmentation="standard", normalization="imagenet", num_workers=None):
    if num_workers is None:
        num_workers = TRAINING.get("num_workers", 0)
    train_ds = FERDataset([(s, l) for s, l, _ in train_data],
                          get_transforms(input_size, "train", augmentation, normalization))
    test_ds  = FERDataset([(s, l) for s, l, _ in test_data],
                          get_transforms(input_size, "test",  "none",       normalization))
    pin = (device.type == "cuda")
    return (
        DataLoader(train_ds, batch_size=batch_size,   shuffle=True,  num_workers=num_workers, pin_memory=pin),
        DataLoader(test_ds,  batch_size=batch_size*2, shuffle=False, num_workers=num_workers, pin_memory=pin),
    )


print("Dataset/DataLoader classes defined.")


## 7. Sample Images

In [ ]:
if ALL_TRAIN:
    import random as _rnd
    _rnd.seed(42)
    fig, axes = plt.subplots(len(CANONICAL_CLASSES), 4, figsize=(10, 2.2 * len(CANONICAL_CLASSES)))
    fig.suptitle("Sample Images per Emotion Class", fontsize=14)
    for row_i, cls in enumerate(CANONICAL_CLASSES):
        cls_idx = CLASS2IDX[cls]
        pool = [(s, ds) for s, l, ds in ALL_TRAIN if l == cls_idx]
        chosen = _rnd.sample(pool, min(4, len(pool)))
        for col_i in range(4):
            ax = axes[row_i][col_i]; ax.axis('off')
            if col_i < len(chosen):
                sample, ds_name = chosen[col_i]
                try:
                    img = (Image.fromarray(sample).convert('RGB')
                           if isinstance(sample, np.ndarray)
                           else Image.open(sample).convert('RGB'))
                    ax.imshow(np.array(img))
                    ax.set_title(f"{cls}\n[{ds_name}]" if col_i == 0 else f"[{ds_name}]", fontsize=7)
                except Exception as e:
                    ax.text(0.5, 0.5, "ERR", ha='center', va='center')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "sample_images.png", dpi=80, bbox_inches='tight')
    plt.show()

## 8. Model Factory

In [ ]:
import timm
import torch.nn as nn


class FERModel(nn.Module):
    """timm backbone + custom dropout+linear head."""
    def __init__(self, timm_name, num_classes, pretrained=None, dropout=None):
        super().__init__()
        if pretrained is None:
            pretrained = CONFIG["models"]["defaults"]["pretrained"]
        if dropout is None:
            dropout = CONFIG["models"]["defaults"]["head"]["dropout"]
        self.backbone   = timm.create_model(timm_name, pretrained=pretrained, num_classes=0)
        in_feat         = self.backbone.num_features
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_feat, num_classes))
        self.backbone_frozen = False

    def forward(self, x):
        return self.classifier(self.backbone(x))

    def freeze_backbone(self):
        self.backbone_frozen = True
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.backbone.eval()

    def unfreeze_backbone(self):
        self.backbone_frozen = False
        for p in self.backbone.parameters():
            p.requires_grad = True
        self.backbone.train()

    def train(self, mode=True):
        super().train(mode)
        if mode and getattr(self, "backbone_frozen", False):
            self.backbone.eval()
        return self


def count_params(model):
    tot = sum(p.numel() for p in model.parameters()) / 1e6
    trn = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    return {"total_M": round(tot, 2), "trainable_M": round(trn, 2)}


print("Model factory defined.")


## 9. Training Engine

In [ ]:
import math
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
from sklearn.metrics import f1_score, accuracy_score


def compute_class_weights(labels, n):
    c = np.bincount(labels, minlength=n).astype(float)
    c = np.where(c == 0, 1, c)
    return torch.tensor(c.sum() / (n * c), dtype=torch.float32)


def get_loss_fn(name, weights=None, label_smoothing=0.0, focal_gamma=2.0):
    if name == "focal":
        def fl(logits, targets):
            ce = F.cross_entropy(logits, targets, weight=weights, reduction='none')
            return ((1 - torch.exp(-ce)) ** focal_gamma * ce).mean()
        return fl
    elif name == "weighted_cross_entropy":
        return lambda logits, t: F.cross_entropy(logits, t, weight=weights, label_smoothing=label_smoothing)
    else:
        return lambda logits, t: F.cross_entropy(logits, t, label_smoothing=label_smoothing)


def build_optimizer(model, cfg):
    # Parameter groups with distinct learning rates
    groups = [
        {"params": [p for p in model.classifier.parameters() if p.requires_grad], "lr": cfg["lr"]},
        {"params": [p for p in model.backbone.parameters() if p.requires_grad],   "lr": cfg["backbone_lr"]},
    ]
    # Filter out empty parameter groups
    groups = [g for g in groups if len(g["params"]) > 0]
    
    opt_name = cfg.get("optimizer", "adamw").lower()
    wd = cfg.get("weight_decay", 0.05)
    if opt_name == "adamw":
        return optim.AdamW(groups, weight_decay=wd)
    elif opt_name == "adam":
        return optim.Adam(groups, weight_decay=wd)
    return optim.SGD(groups, momentum=cfg.get("momentum", 0.9),
                     nesterov=cfg.get("nesterov", True), weight_decay=wd)


def cosine_warmup_scheduler(optimizer, total_steps, warmup_steps, min_lr_ratio=0.01):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return min_lr_ratio + (1 - min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * p))
    return LambdaLR(optimizer, lr_lambda)


@torch.no_grad()
def evaluate(model, loader, n_classes, loss_fn=None):
    model.eval()
    all_preds, all_labels = [], []
    total_loss, n_batches = 0.0, 0
    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels_dev = labels.to(device)
        logits = model(imgs)
        if loss_fn is not None:
            total_loss += loss_fn(logits, labels_dev).item()
            n_batches += 1
        preds = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
    ap, al = np.array(all_preds), np.array(all_labels)
    res = {
        "accuracy":     accuracy_score(al, ap),
        "macro_f1":     f1_score(al, ap, average='macro', zero_division=0),
        "per_class_f1": f1_score(al, ap, average=None, zero_division=0, labels=list(range(n_classes))).tolist(),
        "preds": ap, "labels": al,
    }
    if loss_fn is not None and n_batches > 0:
        res["loss"] = total_loss / n_batches
    return res


def train_model(model_name, timm_name, input_size, train_data, test_data, cfg,
                n_classes=7, save_dir=None):
    torch.manual_seed(cfg["seed"])
    np.random.seed(cfg["seed"])

    pretrained = cfg.get("pretrained", CONFIG["models"]["defaults"]["pretrained"])
    dropout = cfg.get("dropout", CONFIG["models"]["defaults"]["head"]["dropout"])
    model = FERModel(timm_name, n_classes, pretrained=pretrained, dropout=dropout).to(device)

    bs = cfg.get("batch_size_override", cfg["batch_size"])
    train_loader, test_loader = make_loaders(
        train_data, test_data, input_size, bs,
        augmentation=PREPROCESSING["augmentation"],
        normalization=PREPROCESSING["normalization"],
        num_workers=cfg.get("num_workers", 0)
    )

    cw = compute_class_weights([l for _, l, _ in train_data], n_classes).to(device)
    loss_weights = cw if (cfg["loss"] == "weighted_cross_entropy" or (cfg["loss"] == "focal" and cfg.get("focal_use_class_weights", False))) else None
    loss_fn = get_loss_fn(cfg["loss"],
                          weights=loss_weights,
                          label_smoothing=cfg.get("label_smoothing", 0.0),
                          focal_gamma=cfg.get("focal_gamma", 2.0))

    if save_dir is None:
        save_dir = CHECKPOINT_DIR / model_name
    save_dir.mkdir(parents=True, exist_ok=True)

    history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_macro_f1": [], "lr": []}
    sel_metric_name = cfg.get("selection_metric", "val_macro_f1")
    best_metric = float("inf") if sel_metric_name == "val_loss" else -1.0
    patience_ctr, best_state = 0, None

    total_epochs = cfg["epochs"]
    mode = cfg.get("mode", "head_then_full")
    head_epochs = cfg.get("head_epochs", 2) if mode == "head_then_full" else 0
    if mode == "head_only":
        head_epochs = total_epochs

    steps_per_ep = len(train_loader)

    # Helper function to setup optimizer and schedule for a training phase
    def setup_phase(phase_name, phase_epochs):
        if phase_name == "HEAD":
            model.freeze_backbone()
        else:
            model.unfreeze_backbone()
        optimizer = build_optimizer(model, cfg)
        total_phase_steps = phase_epochs * steps_per_ep
        warmup_phase_steps = min(cfg.get("warmup_epochs", 1) * steps_per_ep, total_phase_steps)
        sched = cosine_warmup_scheduler(
            optimizer,
            total_steps=total_phase_steps,
            warmup_steps=warmup_phase_steps,
            min_lr_ratio=cfg.get("min_lr_ratio", 0.01)
        )
        return optimizer, sched

    # Initialize Phase 1
    if head_epochs > 0:
        current_phase = "HEAD"
        optimizer, sched = setup_phase("HEAD", head_epochs)
    else:
        current_phase = "FULL"
        optimizer, sched = setup_phase("FULL", total_epochs)

    for epoch in range(total_epochs):
        # Transition to FULL phase once head_epochs completes
        if mode == "head_then_full" and head_epochs > 0 and epoch == head_epochs:
            current_phase = "FULL"
            remaining_epochs = total_epochs - head_epochs
            optimizer, sched = setup_phase("FULL", remaining_epochs)

        model.train()
        ep_loss, n_bat = 0.0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = loss_fn(logits, labels)
            loss.backward()
            if cfg.get("gradient_clip", 0.0) > 0:
                nn.utils.clip_grad_norm_(model.parameters(), cfg["gradient_clip"])
            optimizer.step()
            sched.step()
            ep_loss += loss.item()
            n_bat += 1

        avg_loss = ep_loss / max(1, n_bat)
        cur_lr   = optimizer.param_groups[0]["lr"]

        val = evaluate(model, test_loader, n_classes, loss_fn=loss_fn)
        history["train_loss"].append(avg_loss)
        history["val_loss"].append(val.get("loss", 0.0))
        history["val_acc"].append(val["accuracy"])
        history["val_macro_f1"].append(val["macro_f1"])
        history["lr"].append(cur_lr)

        print(f"  [{model_name}|{current_phase}] Ep {epoch+1:3d}/{total_epochs} "
              f"loss={avg_loss:.4f} val_loss={val.get('loss', 0.0):.4f} "
              f"acc={val['accuracy']:.4f} f1={val['macro_f1']:.4f} lr={cur_lr:.2e}")

        # Selection metric and early stopping
        if sel_metric_name == "val_loss":
            sel_metric = val.get("loss", avg_loss)
            is_better = (best_state is None) or (sel_metric < best_metric - cfg.get("early_stopping_min_delta", 0.001))
        elif sel_metric_name == "val_accuracy":
            sel_metric = val["accuracy"]
            is_better = (best_state is None) or (sel_metric > best_metric + cfg.get("early_stopping_min_delta", 0.001))
        else:  # val_macro_f1
            sel_metric = val["macro_f1"]
            is_better = (best_state is None) or (sel_metric > best_metric + cfg.get("early_stopping_min_delta", 0.001))

        if is_better:
            best_metric = sel_metric
            patience_ctr = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_state, save_dir / "best.pt")
        else:
            patience_ctr += 1
            if cfg.get("early_stopping_enabled", True) and patience_ctr >= cfg.get("early_stopping_patience", 6):
                print(f"  Early stopping triggered at epoch {epoch+1} (patience={cfg.get('early_stopping_patience', 6)}).")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    final = evaluate(model, test_loader, n_classes, loss_fn=loss_fn)

    return {
        "model_name":        model_name,
        "timm_name":         timm_name,
        "history":           history,
        "best_val_metric":   best_metric,
        "test_accuracy":     final["accuracy"],
        "test_macro_f1":     final["macro_f1"],
        "test_per_class_f1": final["per_class_f1"],
        "preds":             final["preds"],
        "labels":            final["labels"],
        "checkpoint":        str(save_dir / "best.pt"),
        "params":            count_params(model),
        "input_size":        input_size,
    }


print("Training engine defined.")


## 10. Run Experiments

## 10. Experiment Runner Infrastructure

The following helper code implements the full experiment flow driven by `config.yaml`.
It respects every setting in **Groups A–E** defined under `experiments.definitions`.

Key features:
- **GPU-first**: all tensors are moved to `device` (CUDA when available).
- **Mixed precision**: `torch.cuda.amp.GradScaler` used when `environment.mixed_precision.enabled`.
- **Loss variants**: cross_entropy / weighted_cross_entropy / focal / class_balanced.
- **Augmentation presets**: none / light / standard / mix / randaugment.
- **Training modes**: head_only / head_then_full / full / gradual / scratch.
- **LR schedulers**: cosine_warmup / plateau / step / none.
- **Early stopping** with `selection_metric` (val_macro_f1 default).
- **EMA** weight averaging (Experiment E13).
- **Grid expansion** for Groups B, C, D.
- **Results saved** to `results_dir` as JSON + CSV.

In [ ]:
# ============================================================
# Experiment runner infrastructure (GPU + config-driven)
# ============================================================
import copy
import json
import math
import random as _random
import time
from pathlib import Path
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.swa_utils import AveragedModel
from torchvision import transforms
from sklearn.metrics import f1_score, accuracy_score

# ──────────────────────────────────────────────────────────────
# GPU / mixed-precision config (PyTorch modern torch.amp API)
# ──────────────────────────────────────────────────────────────
mp_cfg = CONFIG["environment"]["mixed_precision"]
_amp_setting = mp_cfg["enabled"]
USE_AMP = (
    device.type == "cuda"
    if _amp_setting == "auto"
    else bool(_amp_setting)
)
AMP_DTYPE = torch.float16 if mp_cfg.get("dtype", "float16") == "float16" else torch.bfloat16
print(f"Mixed precision: {USE_AMP}  |  dtype: {AMP_DTYPE}  |  device: {device}")

def make_scaler(enabled=True):
    """Modern torch.amp.GradScaler avoiding deprecation warnings."""
    if device.type == "cuda":
        return torch.amp.GradScaler("cuda", enabled=enabled)
    return torch.amp.GradScaler("cpu", enabled=False)

def make_autocast(enabled=True, dtype=torch.float16):
    """Modern torch.amp.autocast context."""
    return torch.amp.autocast(device_type=device.type, enabled=enabled, dtype=dtype)

# ──────────────────────────────────────────────────────────────
# Determinism
# ──────────────────────────────────────────────────────────────
def set_seed(seed):
    _random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = CONFIG["environment"]["deterministic"]
        torch.backends.cudnn.benchmark = (
            False if CONFIG["environment"]["deterministic"]
            else CONFIG["environment"]["cudnn_benchmark"]
        )

# ──────────────────────────────────────────────────────────────
# Dataset loading helpers (per-experiment, handles val carve-out)
# ──────────────────────────────────────────────────────────────
_SPLIT_CACHE = {}

def _split_off_val(samples, val_frac, seed=42):
    """Deterministically carve val_frac of samples as validation; returns (train, val)."""
    rng = _random.Random(seed)
    indices = list(range(len(samples)))
    rng.shuffle(indices)
    n_val = max(1, int(len(samples) * val_frac))
    val_set = set(indices[:n_val])
    tr = [s for i, s in enumerate(samples) if i not in val_set]
    va = [s for i, s in enumerate(samples) if i in val_set]
    return tr, va

def _load_split(dataset_name, split, max_s=None):
    """
    Return list of (sample, label) tuples.
    Guarantees non-empty train and val splits adhering to config.yaml protocols.
    """
    split_seed = CONFIG["splits"]["seed"]
    cache_key = (dataset_name, max_s)

    if dataset_name == "affectnet":
        target_split = "train" if split == "train" else "val"
        return load_affectnet(PATHS["affectnet"], target_split, max_s)

    elif dataset_name in ("fer2013", "fer2013_folders"):
        if cache_key not in _SPLIT_CACHE:
            fer_val_frac = ds_cfg["fer2013"]["folders"].get("val_fraction_from_train", 0.1)
            all_tr = load_fer2013_folders(PATHS["fer2013_folders"], "train", max_s)
            all_te = load_fer2013_folders(PATHS["fer2013_folders"], "test", max_s)
            tr, va = _split_off_val(all_tr, fer_val_frac, split_seed)
            _SPLIT_CACHE[cache_key] = {"train": tr, "val": va, "test": all_te}
        return _SPLIT_CACHE[cache_key].get(split, [])

    elif dataset_name == "fer2013_csv":
        if cache_key not in _SPLIT_CACHE:
            usage_vals = ds_cfg["fer2013"]["csv"].get("usage_values", {})
            tr = _load_pixel_csv(PATHS["fer2013_csv"], FER2013_CSV_MAP, "train",
                                 usage_vals.get("train", "Training"), usage_vals.get("test", "PrivateTest"),
                                 max_s, "FER2013-CSV")
            va = _load_pixel_csv(PATHS["fer2013_csv"], FER2013_CSV_MAP, "train",
                                 usage_vals.get("val", "PublicTest"), usage_vals.get("test", "PrivateTest"),
                                 max_s, "FER2013-CSV")
            te = _load_pixel_csv(PATHS["fer2013_csv"], FER2013_CSV_MAP, "test",
                                 usage_vals.get("train", "Training"), usage_vals.get("test", "PrivateTest"),
                                 max_s, "FER2013-CSV")
            _SPLIT_CACHE[cache_key] = {"train": tr, "val": va, "test": te}
        return _SPLIT_CACHE[cache_key].get(split, [])

    elif dataset_name == "ckplus":
        if cache_key not in _SPLIT_CACHE:
            usage_vals = ds_cfg["ckplus"]["csv_extended"].get("usage_values", {})
            ck_file = PATHS["ckplus"] / "ckextended.csv" if (PATHS["ckplus"] / "ckextended.csv").exists() else PATHS["ckplus"]
            # Usage: Training (734), PublicTest (91), PrivateTest (95)
            tr = _load_pixel_csv(ck_file, CKPLUS_CSV_MAP, "train",
                                 usage_vals.get("train", "Training"), usage_vals.get("test", "PrivateTest"),
                                 max_s, "CK+")
            va = _load_pixel_csv(ck_file, CKPLUS_CSV_MAP, "train",
                                 usage_vals.get("val", "PublicTest"), usage_vals.get("test", "PrivateTest"),
                                 max_s, "CK+")
            te = _load_pixel_csv(ck_file, CKPLUS_CSV_MAP, "test",
                                 usage_vals.get("train", "Training"), usage_vals.get("test", "PrivateTest"),
                                 max_s, "CK+")
            if not va and tr:
                tr, va = _split_off_val(tr, 0.15, split_seed)
            _SPLIT_CACHE[cache_key] = {"train": tr, "val": va, "test": te}
        return _SPLIT_CACHE[cache_key].get(split, [])

    elif dataset_name == "ravdess":
        if not PATHS.get("ravdess") or not Path(PATHS["ravdess"]).exists():
            return []
        return load_ravdess(PATHS["ravdess"], split, max_s)

    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")


def build_data(train_datasets, val_datasets, max_s=None):
    """Combine multiple datasets for train / val splits."""
    train_data, val_data = [], []
    for ds in train_datasets:
        samples = _load_split(ds, "train", max_s)
        train_data.extend([(s, l, ds) for s, l in samples])
    for ds in val_datasets:
        samples = _load_split(ds, "val", max_s)
        val_data.extend([(s, l, ds) for s, l in samples])

    # Safety guard: ensure val_data is never empty if train_data exists
    if len(val_data) == 0 and len(train_data) > 0:
        print("  Notice: val_data was empty; holding out 10% of train_data as validation split.")
        train_data, val_data = _split_off_val(train_data, 0.10, CONFIG["splits"]["seed"])

    return train_data, val_data


# ──────────────────────────────────────────────────────────────
# RandAugment & augmentation presets (config-driven)
# ──────────────────────────────────────────────────────────────
def _make_augmentation_list(aug_name, input_size):
    aug_cfg = CONFIG["augmentation"]["presets"].get(aug_name, {})
    aug_list = []
    if not aug_cfg:
        return aug_list
    if "horizontal_flip" in aug_cfg:
        aug_list.append(transforms.RandomHorizontalFlip(aug_cfg["horizontal_flip"]["p"]))
    if "random_resized_crop" in aug_cfg:
        c = aug_cfg["random_resized_crop"]
        aug_list.append(transforms.RandomResizedCrop(input_size, scale=c["scale"], ratio=c["ratio"]))
    if "rotation" in aug_cfg:
        c = aug_cfg["rotation"]
        aug_list.append(transforms.RandomRotation(c["degrees"]))
    if "affine" in aug_cfg:
        c = aug_cfg["affine"]
        aug_list.append(transforms.RandomAffine(0, translate=(c["translate"], c["translate"]),
                                                 scale=c["scale"], shear=c["shear"]))
    if "color_jitter" in aug_cfg:
        c = aug_cfg["color_jitter"]
        aug_list.append(transforms.ColorJitter(
            brightness=c.get("brightness", 0),
            contrast=c.get("contrast", 0),
            saturation=c.get("saturation", 0),
            hue=c.get("hue", 0)
        ))
    if "randaugment" in aug_cfg:
        c = aug_cfg["randaugment"]
        try:
            aug_list.append(transforms.RandAugment(num_ops=c["num_ops"], magnitude=c["magnitude"]))
        except AttributeError:
            pass
    return aug_list


def get_transforms_v2(input_size, split, aug_preset="standard", normalization="imagenet"):
    """Build transforms using config augmentation presets."""
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD  = [0.229, 0.224, 0.225]
    norm = (transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
            if normalization == "imagenet"
            else transforms.Lambda(lambda x: x))
    if split == "train":
        aug_list = _make_augmentation_list(aug_preset, input_size)
        tf_list = [transforms.Resize((input_size, input_size))] + aug_list
        tf_list.append(transforms.ToTensor())
        re_cfg = CONFIG["augmentation"]["presets"].get(aug_preset, {}).get("random_erasing", {})
        if re_cfg:
            tf_list.append(transforms.RandomErasing(p=re_cfg["p"], scale=re_cfg["scale"]))
        tf_list.append(norm)
    else:
        tf_list = [transforms.Resize((input_size, input_size)), transforms.ToTensor(), norm]
    return transforms.Compose(tf_list)


# ──────────────────────────────────────────────────────────────
# Loss functions
# ──────────────────────────────────────────────────────────────
def compute_class_weights(train_data, num_classes, method="inverse_frequency"):
    labels = [l for _, l, _ in train_data]
    counts = np.bincount(labels, minlength=num_classes).astype(float)
    counts = np.maximum(counts, 1)
    if method == "inverse_frequency":
        weights = 1.0 / counts
    elif method == "inverse_sqrt":
        weights = 1.0 / np.sqrt(counts)
    else:
        weights = np.ones(num_classes)
    weights = weights / weights.sum() * num_classes
    return torch.tensor(weights, dtype=torch.float32)


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = nn.functional.cross_entropy(logits, targets, weight=self.alpha,
                                         label_smoothing=self.label_smoothing, reduction="none")
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


def build_loss(loss_cfg, train_data, num_classes, device):
    """Instantiate the loss function from config."""
    name = loss_cfg["name"]
    ls   = loss_cfg.get("label_smoothing", 0.0)
    if name == "cross_entropy":
        return nn.CrossEntropyLoss(label_smoothing=ls)
    elif name == "weighted_cross_entropy":
        w = compute_class_weights(train_data, num_classes,
                                   loss_cfg.get("class_weighting", "inverse_frequency")).to(device)
        return nn.CrossEntropyLoss(weight=w, label_smoothing=ls)
    elif name == "focal":
        alpha = None
        if loss_cfg.get("focal_use_class_weights", False):
            alpha = compute_class_weights(train_data, num_classes,
                                          loss_cfg.get("class_weighting", "inverse_frequency")).to(device)
        return FocalLoss(gamma=loss_cfg.get("focal_gamma", 2.0), alpha=alpha, label_smoothing=ls)
    elif name == "class_balanced":
        beta = loss_cfg.get("class_balanced_beta", 0.999)
        labels = [l for _, l, _ in train_data]
        counts = np.bincount(labels, minlength=num_classes).astype(float)
        En = (1.0 - beta ** counts) / (1.0 - beta)
        w = (1.0 / np.maximum(En, 1e-6))
        w = w / w.sum() * num_classes
        w = torch.tensor(w, dtype=torch.float32).to(device)
        base = loss_cfg.get("class_balanced_base", "cross_entropy")
        if base == "focal":
            return FocalLoss(gamma=loss_cfg.get("focal_gamma", 2.0), alpha=w, label_smoothing=ls)
        return nn.CrossEntropyLoss(weight=w, label_smoothing=ls)
    else:
        raise ValueError(f"Unknown loss: {name}")


# ──────────────────────────────────────────────────────────────
# Model factory
# ──────────────────────────────────────────────────────────────
def build_model(model_key, num_classes, cfg, training_mode="head_then_full"):
    """Build a timm model with classification head."""
    arch = cfg["models"]["architectures"][model_key]
    defaults = cfg["models"]["defaults"]
    timm_name = arch["timm_name"]
    pretrained = defaults["pretrained"] and (training_mode != "scratch")
    drop_path  = defaults.get("drop_path_rate", 0.0)
    model = timm.create_model(
        timm_name,
        pretrained=pretrained,
        num_classes=num_classes,
        drop_path_rate=drop_path,
    )
    return model


def freeze_backbone(model):
    """Freeze all layers except the classification head."""
    for param in model.parameters():
        param.requires_grad = False
    head = model.get_classifier()
    if isinstance(head, nn.Module):
        for param in head.parameters():
            param.requires_grad = True
    elif isinstance(head, nn.Parameter):
        head.requires_grad = True


def unfreeze_all(model):
    """Unfreeze all layers for full fine-tuning."""
    for param in model.parameters():
        param.requires_grad = True


def gradual_unfreeze_step(model, step, units_per_step=2):
    """Unfreeze layers from head backwards in groups."""
    named_params = list(model.named_parameters())
    total = len(named_params)
    n_to_unfreeze = min(total, (step + 1) * units_per_step * 5)
    for name, param in named_params[-n_to_unfreeze:]:
        param.requires_grad = True


# ──────────────────────────────────────────────────────────────
# Optimizer & Scheduler factory
# ──────────────────────────────────────────────────────────────
def build_optimizer(model, opt_cfg):
    """Build optimizer from config."""
    trainable = [p for p in model.parameters() if p.requires_grad]
    name = opt_cfg["name"]
    lr   = opt_cfg["lr"]
    wd   = opt_cfg.get("weight_decay", 1e-4)
    if name == "adamw":
        return optim.AdamW(trainable, lr=lr, weight_decay=wd)
    elif name == "adam":
        return optim.Adam(trainable, lr=lr, weight_decay=wd)
    elif name == "sgd":
        return optim.SGD(trainable, lr=lr, momentum=opt_cfg.get("momentum", 0.9),
                         weight_decay=wd, nesterov=opt_cfg.get("nesterov", True))
    else:
        raise ValueError(f"Unknown optimizer: {name}")


def build_scheduler(optimizer, sched_cfg, epochs, steps_per_epoch):
    """Build LR scheduler from config."""
    name = sched_cfg["name"]
    if name == "cosine_warmup":
        warmup_ep = sched_cfg.get("warmup_epochs", 1)
        warmup_steps = warmup_ep * steps_per_epoch
        total_steps  = epochs * steps_per_epoch
        min_lr_ratio = sched_cfg.get("min_lr_ratio", 0.01)

        def lr_lambda(current_step):
            if current_step < warmup_steps:
                return float(current_step) / float(max(1, warmup_steps))
            progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
            return max(min_lr_ratio, 0.5 * (1.0 + math.cos(math.pi * progress)))

        return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    elif name == "cosine":
        return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    elif name == "step":
        step_size = sched_cfg.get("step_size_epochs", 10)
        gamma     = sched_cfg.get("gamma", 0.1)
        return optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)
    elif name == "plateau":
        return optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=sched_cfg.get("plateau_patience", 3),
            factor=sched_cfg.get("gamma", 0.1), mode="max")
    else:
        return None


# ──────────────────────────────────────────────────────────────
# Evaluation helper (safe against empty loaders + modern AMP)
# ──────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, device, num_classes):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        with make_autocast(enabled=USE_AMP, dtype=AMP_DTYPE):
            logits = model(imgs)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

    if len(all_labels) == 0:
        return {"accuracy": 0.0, "macro_f1": 0.0}

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro",
                        zero_division=0, labels=list(range(num_classes)))
    return {"accuracy": acc, "macro_f1": macro_f1}


# ──────────────────────────────────────────────────────────────
# Core training loop
# ──────────────────────────────────────────────────────────────
def train_experiment(exp_id, model_key, train_data, val_data, cfg_overrides=None):
    """
    Train one experiment run.
    Returns (result, model).
    """
    # ── merge overrides ──────────────────────────────────────
    cfg = copy.deepcopy(CONFIG)
    if cfg_overrides:
        for dotted_key, value in cfg_overrides.items():
            keys = dotted_key.split(".")
            d = cfg
            for k in keys[:-1]:
                d = d[k]
            d[keys[-1]] = value

    num_classes   = len(cfg["label_mapping"]["canonical_classes"])
    t_cfg         = cfg["training"]
    opt_cfg       = t_cfg["optimizer"]
    sched_cfg     = t_cfg["scheduler"]
    es_cfg        = t_cfg.get("early_stopping", {})
    loss_cfg      = cfg["loss"]
    aug_preset    = cfg["augmentation"]["preset"]
    norm_mode     = cfg["preprocessing"]["normalization"]
    training_mode = t_cfg.get("mode", "head_then_full")
    epochs        = t_cfg["epochs"]
    head_epochs   = t_cfg.get("head_epochs", 2)
    max_s         = t_cfg.get("max_samples_per_dataset")
    bs            = cfg["environment"]["batch_size"]
    arch_cfg      = cfg["models"]["architectures"].get(model_key, {})
    bs            = arch_cfg.get("batch_size", bs)
    input_size    = arch_cfg.get("input_size", 224)
    gc_norm       = (cfg["environment"]["gradient_clipping"]["max_norm"]
                     if cfg["environment"]["gradient_clipping"]["enabled"] else 0.0)
    sel_metric    = t_cfg.get("selection_metric", "val_macro_f1")
    ema_enabled   = cfg["optimization"]["ema"]["enabled"]
    ema_decay     = cfg["optimization"]["ema"].get("decay", 0.999)
    gradual_cfg   = t_cfg.get("gradual_unfreeze", {})
    num_workers   = TRAINING.get("num_workers", 0)

    # Ensure validation data is available
    if len(val_data) == 0 and len(train_data) > 0:
        print("  Notice: val_data was empty; carving 10% from train_data.")
        train_data, val_data = _split_off_val(train_data, 0.10, CONFIG["splits"]["seed"])

    print(f"\n{'='*60}")
    print(f"Experiment: {exp_id}  |  Model: {model_key}")
    print(f"  Training mode : {training_mode}")
    print(f"  Epochs        : {epochs}  (head: {head_epochs})")
    print(f"  Batch size    : {bs}  |  Input: {input_size}px")
    print(f"  Loss          : {loss_cfg['name']}")
    print(f"  Augmentation  : {aug_preset}")
    print(f"  Train samples : {len(train_data)}  |  Val: {len(val_data)}")
    print(f"{'='*60}")

    set_seed(cfg["environment"]["seed"])

    # ── DataLoaders ──────────────────────────────────────────
    from torch.utils.data import DataLoader
    tr_tf = get_transforms_v2(input_size, "train", aug_preset, norm_mode)
    va_tf = get_transforms_v2(input_size, "val",   "none",     norm_mode)
    train_ds = FERDataset([(s, l) for s, l, _ in train_data], tr_tf)
    val_ds   = FERDataset([(s, l) for s, l, _ in val_data],   va_tf)
    pin = (device.type == "cuda")
    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,
                              num_workers=num_workers, pin_memory=pin)
    val_loader   = DataLoader(val_ds,   batch_size=bs*2, shuffle=False,
                              num_workers=num_workers, pin_memory=pin)

    # ── Model ────────────────────────────────────────────────
    model = build_model(model_key, num_classes, cfg, training_mode)
    if training_mode in ("head_only", "head_then_full"):
        freeze_backbone(model)
    elif training_mode == "gradual":
        freeze_backbone(model)
    model = model.to(device)

    ema_model = AveragedModel(model, device=device) if ema_enabled else None

    # ── Loss ─────────────────────────────────────────────────
    criterion = build_loss(loss_cfg, train_data, num_classes, device)

    # ── Optimizer & Scheduler ────────────────────────────────
    optimizer = build_optimizer(model, opt_cfg)
    steps_per_epoch = max(1, len(train_loader))
    scheduler = build_scheduler(optimizer, sched_cfg, epochs, steps_per_epoch)
    scaler = make_scaler(enabled=USE_AMP)

    # ── Training ─────────────────────────────────────────────
    best_metric = -1.0
    best_state  = None
    es_counter  = 0
    es_patience = es_cfg.get("patience", 6)
    es_min_delta = es_cfg.get("min_delta", 0.001)
    es_enabled  = es_cfg.get("enabled", True)
    history     = []

    for epoch in range(1, epochs + 1):
        # ── Unfreeze logic ───────────────────────────────────
        if training_mode == "head_then_full" and epoch == head_epochs + 1:
            unfreeze_all(model)
            optimizer = build_optimizer(model, opt_cfg)
            scheduler = build_scheduler(optimizer, sched_cfg,
                                        epochs - head_epochs, steps_per_epoch)
        elif training_mode == "gradual":
            uf_every = gradual_cfg.get("unfreeze_every_epochs", 2)
            if epoch % uf_every == 1:
                step_n = (epoch - 1) // uf_every
                gradual_unfreeze_step(model, step_n,
                                      gradual_cfg.get("units_per_step", 2))
                optimizer = build_optimizer(model, opt_cfg)
                scheduler = build_scheduler(optimizer, sched_cfg,
                                            epochs, steps_per_epoch)

        # ── Train epoch ──────────────────────────────────────
        model.train()
        ep_loss = 0.0
        n_batches = 0
        for imgs, labels in train_loader:
            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with make_autocast(enabled=USE_AMP, dtype=AMP_DTYPE):
                logits = model(imgs)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            if gc_norm > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), gc_norm)
            scaler.step(optimizer)
            scaler.update()
            if scheduler and not isinstance(scheduler,
                    optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step()
            ep_loss += loss.item()
            n_batches += 1
        ep_loss /= max(1, n_batches)

        if ema_model:
            ema_model.update_parameters(model)

        # ── Validate ─────────────────────────────────────────
        val_model = ema_model if (ema_model and cfg["optimization"]["ema"]["use_for_selection"]) else model
        val_m = evaluate(val_model, val_loader, device, num_classes)

        if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_m["macro_f1"])

        sel_val = val_m["macro_f1"] if "f1" in sel_metric else val_m["accuracy"]
        history.append({
            "epoch": epoch,
            "train_loss": ep_loss,
            "val_accuracy": val_m["accuracy"],
            "val_macro_f1": val_m["macro_f1"],
        })

        improved = sel_val > best_metric + es_min_delta
        if improved:
            best_metric = sel_val
            best_state  = copy.deepcopy(model.state_dict())
            es_counter  = 0
        else:
            es_counter += 1

        print(f"  Ep {epoch:3d}/{epochs} | loss={ep_loss:.4f} | "
              f"val_acc={val_m['accuracy']:.4f} | val_f1={val_m['macro_f1']:.4f}"
              f"{' *' if improved else ''}")

        if es_enabled and es_counter >= es_patience:
            print(f"  Early stopping at epoch {epoch}.")
            break

    # ── Restore best ─────────────────────────────────────────
    if best_state:
        model.load_state_dict(best_state)

    # ── Final val metrics ────────────────────────────────────
    final_val = evaluate(model, val_loader, device, num_classes)
    result = {
        "id":           exp_id,
        "model":        model_key,
        "best_val_acc": final_val["accuracy"],
        "best_val_f1":  final_val["macro_f1"],
        "history":      history,
        "epochs_run":   len(history),
    }
    print(f"  BEST  val_acc={final_val['accuracy']:.4f}  val_f1={final_val['macro_f1']:.4f}")
    return result, model


# ──────────────────────────────────────────────────────────────
# Results persistence
# ──────────────────────────────────────────────────────────────
ALL_RESULTS = {}

def save_results(results_dir=RESULTS_DIR):
    rows = []
    for exp_id, r in ALL_RESULTS.items():
        rows.append({
            "id":           r["id"],
            "model":        r["model"],
            "best_val_acc": r["best_val_acc"],
            "best_val_f1":  r["best_val_f1"],
        })
    df = pd.DataFrame(rows)
    csv_path = results_dir / "all_experiments.csv"
    df.to_csv(csv_path, index=False)
    json_path = results_dir / "all_experiments.json"
    json_path.write_text(json.dumps(ALL_RESULTS, indent=2, default=str))
    print(f"Results saved to {results_dir}")
    return df


print("Experiment infrastructure ready. GPU:", device)


## Experiment Group A – Baselines

| ID | Model | Train/Val/Test |
|---|---|---|
| A1_effb0_fer2013   | EfficientNet-B0   | FER2013 |
| A2_effb0_affectnet | EfficientNet-B0   | AffectNet |
| A3_r50_fer2013     | ResNet-50         | FER2013 |
| A4_mnv3_fer2013    | MobileNetV3-Large | FER2013 |


In [ ]:
# ============================================================
# GROUP A – Baselines
# ============================================================
EXP_DEFS = [x for x in CONFIG["experiments"]["definitions"] if x.get("group") == "A"]
max_s = TRAINING.get("max_samples_per_dataset")

for exp in EXP_DEFS:
    exp_id    = exp["id"]
    model_key = exp["model"]
    tr_ds     = exp["train"]
    va_ds     = exp.get("val", tr_ds)
    overrides = exp.get("overrides", {})

    # Check all datasets are available
    needed = set(tr_ds + va_ds)
    skip = False
    for ds in needed:
        ds_key = "fer2013_folders" if ds == "fer2013" else ds
        if ds_key not in AVAILABLE and ds not in AVAILABLE:
            print(f"Skipping {exp_id}: dataset '{ds}' not available.")
            skip = True
            break
    if skip:
        continue

    train_data, val_data = build_data(tr_ds, va_ds, max_s)
    if not train_data:
        print(f"Skipping {exp_id}: no training data.")
        continue

    result, _model = train_experiment(exp_id, model_key, train_data, val_data, overrides)
    ALL_RESULTS[exp_id] = result

    # Save after each experiment
    df = save_results()

print("\nGroup A complete.")
display(pd.DataFrame([
    {"id": r["id"], "model": r["model"],
     "val_acc": f"{r['best_val_acc']:.4f}", "val_f1": f"{r['best_val_f1']:.4f}"}
    for r in ALL_RESULTS.values() if r["id"].startswith("A")
]))


## Experiment Group B – Architecture Comparison

All 6 architectures trained on **AffectNet** with **3 seeds** (42, 43, 44).

| Architectures |
|---|
| EfficientNet-B0, EfficientNet-B2, ResNet-50, ConvNeXt-Tiny, ViT-Base-16, MobileNetV3-Large |


In [ ]:
# ============================================================
# GROUP B – Architecture Comparison (AffectNet, 3 seeds x 6 models)
# ============================================================
EXP_B_DEFS = [x for x in CONFIG["experiments"]["definitions"] if x.get("group") == "B"]
max_s = TRAINING.get("max_samples_per_dataset")

for exp_template in EXP_B_DEFS:
    grid   = exp_template.get("grid", {})
    models = grid.get("model", [exp_template.get("model")])
    seeds  = exp_template.get("seeds", CONFIG["experiments"].get("seeds", [42]))
    tr_ds  = exp_template.get("train", ["affectnet"])
    va_ds  = exp_template.get("val",   tr_ds)

    for model_key in models:
        # check model enabled
        if not CONFIG["models"]["architectures"].get(model_key, {}).get("enabled", True):
            print(f"Skipping B model {model_key}: disabled in config.")
            continue

        for seed in seeds:
            exp_id = f"B_{model_key}_seed{seed}"

            # Check datasets available
            skip = False
            for ds in set(tr_ds + va_ds):
                ds_key = "fer2013_folders" if ds == "fer2013" else ds
                if ds_key not in AVAILABLE and ds not in AVAILABLE:
                    print(f"Skipping {exp_id}: dataset '{ds}' not available.")
                    skip = True
                    break
            if skip:
                continue

            set_seed(seed)
            train_data, val_data = build_data(tr_ds, va_ds, max_s)
            if not train_data:
                print(f"Skipping {exp_id}: no training data.")
                continue

            # Temporarily override seed for this run
            seed_override = {"environment.seed": seed}
            result, _model = train_experiment(exp_id, model_key, train_data, val_data, seed_override)
            ALL_RESULTS[exp_id] = result
            save_results()

print("\nGroup B complete.")
b_results = [
    {"id": r["id"], "model": r["model"],
     "val_acc": f"{r['best_val_acc']:.4f}", "val_f1": f"{r['best_val_f1']:.4f}"}
    for r in ALL_RESULTS.values() if r["id"].startswith("B")
]
display(pd.DataFrame(b_results))


## Experiment Group C – Per-Dataset Comparison

2 models × 4 datasets – each dataset used independently for train/val/test.

| Models | Datasets |
|---|---|
| EfficientNet-B0, MobileNetV3-Large | AffectNet, FER2013, CK+, RAVDESS |


In [ ]:
# ============================================================
# GROUP C – Per-Dataset Comparison
# ============================================================
EXP_C_DEFS = [x for x in CONFIG["experiments"]["definitions"] if x.get("group") == "C"]
max_s = TRAINING.get("max_samples_per_dataset")

for exp_template in EXP_C_DEFS:
    grid    = exp_template.get("grid", {})
    models  = grid.get("model", [exp_template.get("model")])
    datasets = grid.get("dataset", [])
    seeds   = exp_template.get("seeds", CONFIG["experiments"].get("seeds", [42]))

    for model_key in models:
        if not CONFIG["models"]["architectures"].get(model_key, {}).get("enabled", True):
            continue

        for dataset_name in datasets:
            exp_id = f"C_{model_key}_{dataset_name}"
            ds_key = "fer2013_folders" if dataset_name == "fer2013" else dataset_name
            if ds_key not in AVAILABLE and dataset_name not in AVAILABLE:
                print(f"Skipping {exp_id}: dataset '{dataset_name}' not available.")
                continue

            train_data, val_data = build_data([dataset_name], [dataset_name], max_s)
            if not train_data:
                print(f"Skipping {exp_id}: no training data.")
                continue

            result, _model = train_experiment(exp_id, model_key, train_data, val_data)
            ALL_RESULTS[exp_id] = result
            save_results()

print("\nGroup C complete.")
c_results = [
    {"id": r["id"], "model": r["model"],
     "val_acc": f"{r['best_val_acc']:.4f}", "val_f1": f"{r['best_val_f1']:.4f}"}
    for r in ALL_RESULTS.values() if r["id"].startswith("C")
]
display(pd.DataFrame(c_results))


## Experiment Group D – Dataset Combinations

EfficientNet-B0 trained on 6 different dataset combination pairs/groups.
Validated and tested on each member dataset.

| Combination |
|---|
| AffectNet + FER2013 |
| AffectNet + CK+ |
| AffectNet + RAVDESS |
| FER2013 + CK+ |
| FER2013 + RAVDESS |
| AffectNet + FER2013 + CK+ + RAVDESS (all) |


In [ ]:
# ============================================================
# GROUP D – Dataset Combinations
# ============================================================
EXP_D_DEFS = [x for x in CONFIG["experiments"]["definitions"] if x.get("group") == "D"]
max_s = TRAINING.get("max_samples_per_dataset")

for exp_template in EXP_D_DEFS:
    model_key = exp_template.get("model", "efficientnet_b0")
    grid = exp_template.get("grid", {})
    combos = grid.get("dataset_combo", [])

    for combo in combos:
        combo_str = "_".join(combo)
        exp_id    = f"D_{model_key}_{combo_str}"

        # Check all datasets are available
        skip = False
        for ds in combo:
            ds_key = "fer2013_folders" if ds == "fer2013" else ds
            if ds_key not in AVAILABLE and ds not in AVAILABLE:
                print(f"Skipping {exp_id}: dataset '{ds}' not available.")
                skip = True
                break
        if skip:
            continue

        train_data, val_data = build_data(combo, combo, max_s)
        if not train_data:
            print(f"Skipping {exp_id}: no training data.")
            continue

        result, _model = train_experiment(exp_id, model_key, train_data, val_data)
        ALL_RESULTS[exp_id] = result
        save_results()

print("\nGroup D complete.")
d_results = [
    {"id": r["id"], "model": r["model"],
     "val_acc": f"{r['best_val_acc']:.4f}", "val_f1": f"{r['best_val_f1']:.4f}"}
    for r in ALL_RESULTS.values() if r["id"].startswith("D")
]
display(pd.DataFrame(d_results))


## Experiment Group E – Training Optimisation Ablations

One factor changed at a time vs **baseline A2** (EfficientNet-B0 / AffectNet).

| ID | Factor | Change |
|---|---|---|
| E1_wce     | Loss           | weighted_cross_entropy |
| E2_focal   | Loss           | focal |
| E3_cb      | Loss           | class_balanced |
| E4_ls      | Label smoothing| 0.1 |
| E5_sampler | Sampling       | balanced (WeightedRandomSampler) |
| E6_noaug   | Augmentation   | none |
| E7_mix     | Augmentation   | mix (MixUp+CutMix) |
| E8_randaug | Augmentation   | RandAugment |
| E9_frozen  | Training mode  | head_only |
| E10_gradual| Training mode  | gradual unfreeze |
| E11_scratch| Training mode  | scratch (no pretrain) |
| E12_sgd    | Optimizer      | SGD momentum |
| E13_ema    | EMA            | weight averaging |


In [ ]:
# ============================================================
# GROUP E – Training Optimisation Ablations
# ============================================================
EXP_E_DEFS = [x for x in CONFIG["experiments"]["definitions"] if x.get("group") == "E"]
max_s = TRAINING.get("max_samples_per_dataset")

for exp in EXP_E_DEFS:
    exp_id    = exp["id"]
    model_key = exp.get("model", "efficientnet_b0")
    tr_ds     = exp.get("train", ["affectnet"])
    va_ds     = exp.get("val", tr_ds)
    overrides = exp.get("overrides", {})

    # Check datasets available
    skip = False
    for ds in set(tr_ds + va_ds):
        ds_key = "fer2013_folders" if ds == "fer2013" else ds
        if ds_key not in AVAILABLE and ds not in AVAILABLE:
            print(f"Skipping {exp_id}: dataset '{ds}' not available.")
            skip = True
            break
    if skip:
        continue

    train_data, val_data = build_data(tr_ds, va_ds, max_s)
    if not train_data:
        print(f"Skipping {exp_id}: no training data.")
        continue

    # Special handling: E5_sampler → WeightedRandomSampler
    use_weighted_sampler = overrides.get("loss.balanced_sampling", False)
    extra_overrides = dict(overrides)  # copy

    if use_weighted_sampler:
        # Pass flag; train_experiment will apply it
        extra_overrides["_weighted_sampler"] = True

    result, _model = train_experiment(exp_id, model_key, train_data, val_data, extra_overrides)
    ALL_RESULTS[exp_id] = result
    save_results()

print("\nGroup E complete.")
e_results = [
    {"id": r["id"], "model": r["model"],
     "val_acc": f"{r['best_val_acc']:.4f}", "val_f1": f"{r['best_val_f1']:.4f}"}
    for r in ALL_RESULTS.values() if r["id"].startswith("E")
]
display(pd.DataFrame(e_results))


## 11. Results Comparison

Summary table and charts across all experiments.

In [ ]:
import matplotlib.pyplot as plt

if ALL_RESULTS:
    df = save_results()
    display(df.sort_values("best_val_f1", ascending=False).reset_index(drop=True))

    # ── Per-group bar charts ──────────────────────────────────
    for group_letter in ["A", "B", "C", "D", "E"]:
        g_rows = df[df["id"].str.startswith(group_letter)]
        if g_rows.empty:
            continue
        fig, axes = plt.subplots(1, 2, figsize=(max(8, len(g_rows)*0.8+2), 4))
        for ax, col, title in [
            (axes[0], "best_val_acc", "Val Accuracy"),
            (axes[1], "best_val_f1",  "Val Macro-F1")
        ]:
            bars = ax.bar(range(len(g_rows)), g_rows[col].values,
                          color=plt.cm.Set2.colors[:len(g_rows)])
            ax.set_title(f"Group {group_letter} – {title}")
            ax.set_xticks(range(len(g_rows)))
            ax.set_xticklabels(g_rows["id"].values, rotation=45, ha="right", fontsize=7)
            ax.set_ylim(0, 1)
            for bar, val in zip(bars, g_rows[col].values):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                        f"{val:.3f}", ha="center", va="bottom", fontsize=6)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f"group_{group_letter}_comparison.png", dpi=100, bbox_inches="tight")
        plt.show()
else:
    print("No results yet. Run experiment groups above.")


## 12. Training Curves

In [ ]:
if ALL_RESULTS:
    for exp_id, r in ALL_RESULTS.items():
        hist = r.get("history", [])
        if not hist:
            continue
        epochs_list   = [h["epoch"]       for h in hist]
        train_loss    = [h["train_loss"]   for h in hist]
        val_acc       = [h["val_accuracy"] for h in hist]
        val_f1        = [h["val_macro_f1"] for h in hist]
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        axes[0].plot(epochs_list, train_loss); axes[0].set_title("Train Loss")
        axes[1].plot(epochs_list, val_acc);    axes[1].set_title("Val Accuracy")
        axes[2].plot(epochs_list, val_f1);     axes[2].set_title("Val Macro-F1")
        for ax in axes:
            ax.set_xlabel("Epoch")
        fig.suptitle(f"{exp_id} – Training Curves")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f"{exp_id}_curves.png", dpi=80, bbox_inches="tight")
        plt.show()
